# 3. Free-text questions

Analysis in four steps:

1. **Inventory**: who wrote what, how much.
2. **Thematic coding**: every answer is assigned one or more themes from a codebook (`src/codebook.py`); the assignments live in `data/coding/*.csv` and are validated here. Edit those CSV files to change a coding, then re-run.
3. **Patterns**: theme frequencies, theme x respondent group, co-occurrence, per-respondent dot matrices.
4. **Triangulation**: transparent keyword counts, frequent terms and (exploratory) sentiment.

> The codings were drafted by an assistant and must be reviewed. For the thesis, document the codebook, have a second person code a subset, and report the agreement.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append(str(Path.cwd().parent))
from src import codebook as C
from src import plots as P
from src import schema as S
from src import textmining as T
from src.data import constraint_coverage, crosstab_ordered, load_clean, select_population
from src.plots import save_figure

PROCESSED_PATH = Path("../data/processed/survey_clean.csv")
TABLES_DIR = Path("../outputs/tables")
INCLUDE_PARTIAL = False
P.SHOW_TITLES = True
pd.set_option("display.max_colwidth", 170)
pd.set_option("display.width", 220)

df_all, sc = load_clean(PROCESSED_PATH)
df = select_population(df_all, sc, include_partial=INCLUDE_PARTIAL)
ROLE_COLORS = dict(zip(S.ROLE_GROUPS, P.GROUP_PALETTE))
SIMILAR_COLORS = {"More different": P.DIVERGING5[0], "About the same": P.DIVERGING5[2], "More similar": P.DIVERGING5[4]}


def by_respondent(column):
    """Respondent attribute indexed by Response ID (for grouping coded answers)."""
    return df.set_index(sc.response_id)[column]


FIELDS = ["process_factors", "similar_reason", "constraint_comments", "anything_else"]
texts = {f: C.open_text(df, sc, f) for f in FIELDS}
print(f"{len(df)} respondents")

## 1. Inventory

In [ ]:
inventory = pd.DataFrame({
    C.FIELD_TITLES[f]: {
        "Respondents answering": t["response_id"].nunique(),
        "Share of respondents": t["response_id"].nunique() / len(df),
        "Answers (text units)": len(t),
        "Median words": t["text"].map(T.word_count).median() if len(t) else np.nan,
        "Total words": sum(T.word_count(x) for x in t["text"]),
    } for f, t in texts.items()
}).T
display(inventory.round(2))

In [ ]:
lengths = pd.concat([t.assign(question=C.FIELD_TITLES[f], words=t["text"].map(T.word_count))
                     for f, t in texts.items() if len(t)])
order = [C.FIELD_TITLES[f] for f, t in texts.items() if len(t)]
fig = P.strip_box(lengths, "words", "question", order=order, title="Length of the free-text answers",
                  xlabel="Words per answer")
save_figure(fig, "open_answer_length")

### Who writes comments?

In [ ]:
writers = set().union(*[set(texts[f]["response_id"]) for f in ["process_factors", "similar_reason", "constraint_comments"]])
df = df.assign(wrote_comment=df[sc.response_id].isin(writers))
for col, groups, name in [("role_group", S.ROLE_GROUPS, "role group"), ("ai_stance_group", S.AI_STANCE_GROUPS, "AI stance")]:
    tab = crosstab_ordered(df, col, "wrote_comment", groups, [True, False]).rename(
        columns={True: "Wrote a comment", False: "No comment"})
    fig = P.stacked_bars(tab, [P.NAVY, "#D6E4F0"], f"Free-text participation by {name}")
    save_figure(fig, f"open_participation_{name.replace(' ', '_').lower()}")

## 2. Codebooks and coding review

Each answer can carry several themes. The review tables list the verbatim answers next to their themes.

In [ ]:
for f in ["process_factors", "similar_reason", "constraint_comments"]:
    print(C.FIELD_TITLES[f])
    display(C.codebook_frame(f))

In [ ]:
matrices, coded = {}, {}
for f in ["process_factors", "similar_reason", "constraint_comments"]:
    coded[f] = C.validate_coding(f, texts[f])
    matrices[f] = C.theme_matrix(f, texts[f])
review = {f: C.review_table(texts[f], matrices[f]) for f in matrices}
display(review["process_factors"])

In [ ]:
display(review["similar_reason"])

In [ ]:
display(review["constraint_comments"])

## 3a. Factors influencing the development process

Which factors shape the choice or adaptation of the development process?

In [ ]:
f, m = "process_factors", matrices["process_factors"]
counts, colors = C.theme_counts(m, f)
fig = P.bar_chart(counts, "Factors influencing the choice of the development process", n=len(m), colors=colors,
                  xlabel="Respondents mentioning the theme", wrap=36)
P.add_legend(fig.axes[0], C.GROUP_COLORS[f])
save_figure(fig, "open_process_factors_themes")

In [ ]:
fig = P.oncoprint(m, col_groups=by_respondent("role_group"), group_colors=ROLE_COLORS, group_order=S.ROLE_GROUPS,
                  row_groups=C.theme_groups(f), row_group_colors=C.GROUP_COLORS[f], strip_title="Role group",
                  title="Process factors mentioned by each respondent")
save_figure(fig, "open_process_factors_oncoprint")

In [ ]:
for col, order, name in [("role_group", S.ROLE_GROUPS, "role"), ("flex_group", S.FLEXIBILITY_GROUPS, "flexibility")]:
    table = C.theme_by_group(m, by_respondent(col), order)
    fig = P.heatmap(table, f"Process factors by {name} group", cbar_label="Respondents", n=len(m), col_wrap=18)
    save_figure(fig, f"open_process_factors_by_{name}")

## 3b. Why will space and other domains become more similar or more different?

Reasons given alongside the closed question (more similar / about the same / more different).

In [ ]:
f, m = "similar_reason", matrices["similar_reason"]
counts, colors = C.theme_counts(m, f)
fig = P.bar_chart(counts, "Reasons for expecting similar or different software engineering", n=len(m),
                  colors=colors, xlabel="Respondents mentioning the theme", wrap=36)
P.add_legend(fig.axes[0], C.GROUP_COLORS[f])
save_figure(fig, "open_similar_reason_themes")

In [ ]:
fig = P.oncoprint(m, col_groups=by_respondent("similar_group"), group_colors=SIMILAR_COLORS,
                  group_order=S.SIMILAR_GROUPS, row_groups=C.theme_groups(f), row_group_colors=C.GROUP_COLORS[f],
                  strip_title="Expected development", title="Reasons mentioned by each respondent")
save_figure(fig, "open_similar_reason_oncoprint")

In [ ]:
table = C.theme_by_group(m, by_respondent("similar_group"), S.SIMILAR_GROUPS)
fig = P.heatmap(table, "Reasons by expected development", cbar_label="Respondents", n=len(m), col_wrap=18)
save_figure(fig, "open_similar_reason_by_expectation")

In [ ]:
tg = C.theme_groups(f)
direction = pd.DataFrame({g: m[[t for t in m.columns if tg[t] == g]].any(axis=1) for g in C.GROUP_COLORS[f]})
table = C.theme_by_group(direction, by_respondent("similar_group"), S.SIMILAR_GROUPS)
fig = P.heatmap(table, "Direction of the reasons given, by expected development", cbar_label="Respondents",
                n=len(m), col_wrap=18)
save_figure(fig, "open_similar_reason_direction")

In [ ]:
fig = P.heatmap(m.astype(int).T @ m.astype(int), "Reasons mentioned together", mask_upper=True,
                cbar_label="Respondents", n=len(m))
save_figure(fig, "open_similar_reason_cooccurrence")

## 3c. Explanations of project constraints

Respondents who selected a constraint could explain why it matters. First: how often did they do so?

In [ ]:
coverage = constraint_coverage(df, sc).drop("Don't know")
coverage = coverage.loc[coverage.sum(axis=1).sort_values(ascending=False, kind="stable").index]
fig = P.stacked_bars(coverage, [P.NAVY, "#D6E4F0"], "Selected constraints and written explanations", normalize=False,
                     xlabel="Respondents who selected the constraint")
save_figure(fig, "open_constraints_coverage")

In [ ]:
f, m = "constraint_comments", matrices["constraint_comments"]
counts, colors = C.theme_counts(m, f)
fig = P.bar_chart(counts, "Why the selected constraints matter", n=len(m), colors=colors, wrap=36,
                  xlabel="Explanations mentioning the theme")
P.add_legend(fig.axes[0], C.GROUP_COLORS[f])
save_figure(fig, "open_constraints_themes")

In [ ]:
table = C.theme_by_constraint(coded[f])
fig = P.heatmap(table, "Explanation themes per constraint", cbar_label="Explanations", n=len(m),
                xlabel="Selected constraint", col_wrap=14)
save_figure(fig, "open_constraints_theme_by_constraint")

## 4. Triangulation: keywords, terms, sentiment

Keyword counts use transparent regular expressions (`src/textmining.py`) and serve as a plausibility check of the manual coding.

In [ ]:
all_texts = list(dict.fromkeys(pd.concat([t["text"] for t in texts.values() if len(t)])))
prevalence = T.keyword_prevalence(all_texts)
fig = P.bar_chart(prevalence, "Concepts mentioned in the free-text answers", n=len(all_texts), colors=P.TEAL,
                  xlabel="Answers mentioning the concept")
save_figure(fig, "open_keyword_prevalence")

In [ ]:
DOMAIN_STOP = {"space", "software", "development", "projects", "project", "years", "new", "sw"}  # too generic here
terms = T.top_terms(all_texts, n=15, min_count=2, extra_stop=DOMAIN_STOP)
fig = P.bar_chart(terms, "Most frequent terms in the free-text answers", n=len(all_texts), colors=P.NAVY,
                  xlabel="Answers containing the term")
save_figure(fig, "open_top_terms")
bigrams = T.top_terms(all_texts, n=12, min_count=2, ngram=2, extra_stop=DOMAIN_STOP)
if len(bigrams):
    fig = P.bar_chart(bigrams, "Most frequent word pairs", n=len(all_texts), colors=P.NAVY,
                      xlabel="Answers containing the pair")
    save_figure(fig, "open_top_bigrams")

### Sentiment (exploratory)

VADER is a general-purpose English lexicon. Technical statements ("strict regulations", "limited resources") score as negative or positive for reasons unrelated to attitude, so treat this as a descriptive side note.

In [ ]:
sentiment = pd.concat([T.analyze_sentiment(t["text"]).assign(question=C.FIELD_TITLES[f])
                       for f, t in texts.items() if len(t)])
order = [C.FIELD_TITLES[f] for f, t in texts.items() if len(t)]
fig = P.strip_box(sentiment, "compound", "question", order=order, ref_lines=[-0.05, 0.05],
                  title="Sentiment of the free-text answers (VADER)", xlabel="Compound score (-1 negative, +1 positive)")
save_figure(fig, "open_sentiment")
display(sentiment.groupby("question")["label"].value_counts().unstack(fill_value=0))

## 5. Quotes and export for the appendix

In [ ]:
for f in matrices:
    print(C.FIELD_TITLES[f])
    display(C.quotes_by_theme(texts[f], matrices[f], per_theme=2))

In [ ]:
TABLES_DIR.mkdir(parents=True, exist_ok=True)
theme_rows = [{"Question": C.FIELD_TITLES[f], "Theme": t, "Group": C.theme_groups(f)[t], "Answers": int(c),
               "Share of answers": round(c / len(m), 3)}
              for f, m in matrices.items() for t, c in m.sum().items()]
with pd.ExcelWriter(TABLES_DIR / "open_text_coding.xlsx") as xl:
    pd.DataFrame(theme_rows).to_excel(xl, sheet_name="theme_counts", index=False)
    for f in matrices:
        review[f].to_excel(xl, sheet_name=f"answers_{f}"[:31], index=False)
        C.codebook_frame(f).to_excel(xl, sheet_name=f"codebook_{f}"[:31], index=False)
        C.quotes_by_theme(texts[f], matrices[f], per_theme=3).to_excel(xl, sheet_name=f"quotes_{f}"[:31], index=False)
print(f"Saved {(TABLES_DIR / 'open_text_coding.xlsx').resolve()}")